### Rete neurale generativa per simulazioni (addestrata su Geant4) - parte generativa

Spostiamoci su gpu (se esiste) e importiamo il modello, i pesi e gli encoders creati durante la parte di training
- gli encoders servono per riconvertire gli output numerici in variabili stringa
- recuperiamo anche il fattore di normalizzazione

In [1]:
import torch
import numpy as np
import pandas as pd
import pickle
from model_architecture import GRUTransformerGenerator
import torch.nn.functional as F
from first_step_distributions import FirstStepSampler
import numpy as np

# Controlla dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# carica encoders e scalers per ottenere i parametri corretti
with open("encoders_scalers.pkl", "rb") as f:
    stuff = pickle.load(f)

le_part = stuff["le_part"]
le_vol = stuff["le_vol"]
le_moth = stuff["le_moth"]
scalers = stuff["scalers"]

filename="../DatiPerNN_BigMixture_1500keV_PulseTube_1e4EInt.txt"
sampler = FirstStepSampler(
    filename=filename,
)

input_dim = 7  # 7 features in input

# Carica il modello
model = GRUTransformerGenerator(
    input_dim=input_dim,
    hidden_dim=64,
    energy_dim=1,
    gru_layers=1,
    nhead=1,
    num_transformer_layers=1,
    dropout=0.4,
    max_chunk_len=100,
    n_volume_classes=len(le_vol.classes_),           
    n_mother_volume_classes=len(le_moth.classes_)  
).to(device)

model.first_step_sampler = sampler
model.load_state_dict(torch.load("best_model_weights.pt", map_location=device))
model.to(device)
model.eval()

print("Modello e trasformazioni caricati con successo!")

Using device: cpu
Modello e trasformazioni caricati con successo!


/home/emanuelemazzola/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


Recuperiamo gli encoder e il fattore di normalizzazione e applichiamolo all'output

In [2]:
def denormalize_sequence(seq_tensor, fs):
    """
    seq_tensor: tensor [N,7] con feature continue già in unità fisiche (MeV, mm)
    fs: FirstStepSampler instance (usato solo per decodificare le etichette)
    """
    seq = seq_tensor.detach().cpu().numpy().copy()

    # --- variabili continue (già in MeV/mm) ---
    col_names = ["Edep(MeV)", "X(mm)", "Y(mm)", "Z(mm)", "Step(mm)"]

    # --- variabili categoriche ---
    vol_idx = seq[:, 5].astype(int)
    moth_idx = seq[:, 6].astype(int)
    try:
        vol_names = fs.le_vol.inverse_transform(vol_idx)
        moth_names = fs.le_moth.inverse_transform(moth_idx)
    except Exception:
        vol_names = vol_idx
        moth_names = moth_idx

    # --- costruisci DataFrame finale ---
    df = pd.DataFrame(seq[:, :5], columns=col_names)
    df["Volume"] = vol_names
    df["MotherVolume"] = moth_names

    return df


Generiamo le particelle e salviamole in un file txt
- Alcuni valori li impostiamo manualmente, per ora non vengono appresi dal modello

In [3]:
from tqdm import tqdm
import gc
import os

# Numero di particelle da generare
n_particles = 10000

# Energia iniziale in MeV
E_init = 1.5

# Parametro: quante particelle generare/scrivere per volta
chunk_size = 500  

output_filename = os.path.basename(filename).replace(".txt", "_generated.txt")

# Intestazioni
headers = ["EventNumber","ParentID","TrackID","ParticleName","ParticleEn(MeV)",
           "Edep(MeV)","X(mm)","Y(mm)","Z(mm)","Step(mm)","Track(mm)",
           "VolumeName","MotherVolumeName","CenterX(mm)","CenterY(mm)","CenterZ(mm)"]

# Crea file con intestazioni vuoto
with open(output_filename, "w") as f:
    f.write("\t".join(headers) + "\n")

event_counter = 0

# Genera e scrive a blocchi con barra di progresso
with tqdm(total=n_particles, desc="Generazione particelle") as pbar:
    while event_counter < n_particles:
        # Numero di particelle in questo blocco
        n_block = min(chunk_size, n_particles - event_counter)

        # Genera il blocco di particelle
        particles = model.generate_multiple_particles(
            energy=E_init,
            n_particles=n_block,
            max_len=100,
            temperature=1.0,
            noise_scale=1.0
        )

        block_rows = []
        for particle_tensor in particles:
            df_phys = denormalize_sequence(particle_tensor, model.first_step_sampler)

            track_length = 0.0
            for _, step in df_phys.iterrows():
                track_length += step["Step(mm)"]
                block_rows.append([
                    event_counter,
                    0,                        # ParentID
                    1,                        # TrackID
                    "e-",                     # ParticleName
                    E_init,                   # Energia iniziale in MeV
                    step["Edep(MeV)"],
                    step["X(mm)"],
                    step["Y(mm)"],
                    step["Z(mm)"],
                    step["Step(mm)"],
                    track_length,
                    step["Volume"],
                    step["MotherVolume"],
                    0, 0, 0                   # CenterX/Y/Z
                ])
            event_counter += 1
            pbar.update(1)

        # Scrivi il blocco in append (senza header)
        df_block = pd.DataFrame(block_rows, columns=headers)
        df_block.to_csv(output_filename, sep="\t", index=False, mode="a", header=False)

        # Libera memoria del blocco
        del df_block, block_rows, particles
        gc.collect()

print(f"Tutte le {n_particles} particelle salvate in {output_filename}")


Generazione particelle:   0%|          | 0/10000 [00:00<?, ?it/s]

Generazione particelle: 100%|██████████| 10000/10000 [03:22<00:00, 49.46it/s]

Tutte le 10000 particelle salvate in DatiPerNN_BigMixture_1500keV_PulseTube_1e4EInt_generated.txt
